## siliconflow

In [48]:
import requests

class DeepSeekChat:
    def __init__(self, start_key=0,model="deepseek-ai/DeepSeek-R1"):
        # deepseek-r1:32b,deepseek-ai/DeepSeek-V3,qwq:latest,Qwen/QwQ-32B,Pro/deepseek-ai/DeepSeek-R1
        self.next_api_key = start_key
        self.api_key_list=["sk-rcnkczywequftoobogneniitnbcfjummgnpfhbuxncbifihd",#yhq
              "sk-pusevrpzhwwsnsnktxlyxiymshdlesiunuitobadmgpsxqfn",#yxf
              "sk-wfjvalfksdrjvtazdywypdfdnidktogcyzzmxxillskqvyzx",#ycg
              "sk-wlcikcifonhhjetbxffcggckwacddbmedugipfckhpmqsuwh",#hgq
              "sk-nffswispohtowcalrfhmvgihwiljtqwmpgkmzjtrnqcdljcp",#zhangyong
              "sk-cbgvymxlwfhagvohubkbmbivzpmdfkgpgyqzftzdkmjbeygt",#13345560769
              "sk-ezraojkfkxxegsbhuriyrnneeubsokbhhzjvvkfxinhgltqt",#19355265824
              "sk-kzjxgalflvfizcjteviigszbbmqmnsxbcavalctitlmizgqd",#19275604712
              ]
        self.model = model
        self.base_url = "https://api.siliconflow.cn/v1/chat/completions"
        # self.base_url = "https://127.0.0.1:11434/v1"

        self.messages = []

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

    def get_response(self, temperature=0.7, max_tokens=16384):
        payload = {
            "model": self.model,
            "messages": self.messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "top_p": 0.6,
            # "stream": 'True',
        }
        self.headers = {
            "Authorization": f"Bearer {self.api_key_list[self.next_api_key]}",
            "Content-Type": "application/json"
        }
        self.next_api_key=(self.next_api_key+1)%len(self.api_key_list)        
        response = requests.post(self.base_url, json=payload, headers=self.headers)
        
        if response.status_code == 200:
            result = response.json()
            self.add_message("assistant", result['choices'][0]['message']['content'])
            return result['choices'][0]['message']['content']
        else:
            raise Exception(f"API Error: {response.status_code},response:{response.text}")

In [49]:
chat1 = DeepSeekChat(start_key=1)
chat1.add_message("system","You are a moveable robot in a 2D map, you task is to find the shortest path to the target position.")
chat1.add_message("user", """The map is defined by border and lines as fellows,
""
Length,Height=15,8
Width1=2
Width2=0.6
border=[[(Length/2,-Height/2-Width1),(Length/2,-Height/2),(Width2/2,-Height/2),(Width2/2,Height/2),(Length/2,Height/2),(Length/2,Height/2+Width1),(-Length/2,Height/2+Width1),(-Length/2,Height/2),(-Width2/2,Height/2),(-Width2/2,-Height/2),(-Length/2,-Height/2),(-Length/2,-Height/2-Width1),(Length/2,-Height/2-Width1)]]
""
border is a list of lines, each line is a wall that the robot can not pass,
the first and last point in lines should be the same, which means the map is a closed map.

Now your are navigating in this map, your start position is (-6,-Height/2-Width1/2), and the target position is (-6,Height/2+Width1/2), 
you can move to the any directions, but you can not pass the walls defined by the border and lines, you need to find the shortest path to the target position, 
you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible,
the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Give me the shortest path you found, and the path should be output as: ""path=[(x0,y0),(x1,y1),...,(xn,yn)] ""
""")
response1 = chat1.get_response()

Exception: API Error: 504,response:{"code":50509,"message":"Model service timeout. Please try again later.","data":null}

In [41]:
print(response1)

"path=[(-6, -5), (0.3,4), (-6,5)]"


In [ ]:
import re
# 提取``` ```之间的内容

# pattern = re.compile(r''''(.*?)'''', re.DOTALL)
pattern = re.compile(r'```(.*?)```', re.DOTALL)
res="""
The shortest path navigates along the left wall to bypass the horizontal barriers at y = -4 and y = 4. Here's the optimal path with minimal waypoints:

```
path=[(-6, -5.0), (-7.5, -5.0), (-7.5, 5.0), (-6, 5.0)]
```

**Explanation:**
1. **Start at (-6, -5):** The robot begins at the specified starting position.
2. **Move left to (-7.5, -5):** This avoids the horizontal wall at y = -4 by staying along the left boundary.
3. **Move up along the left wall to (-7.5, 5):** Ascends vertically without crossing any walls.
4. **Move right to the target (-6, 5):** Directly reaches the target after clearing the upper horizontal wall.

This path ensures no intersections with walls and uses the fewest waypoints.
"""
map_info = pattern.findall(response1)[0]
print(map_info)

In [25]:
from openai import OpenAI
api_key_list=["sk-rcnkczywequftoobogneniitnbcfjummgnpfhbuxncbifihd",#yhq
        "sk-pusevrpzhwwsnsnktxlyxiymshdlesiunuitobadmgpsxqfn",#yxf
        "sk-wfjvalfksdrjvtazdywypdfdnidktogcyzzmxxillskqvyzx",#ycg
        "sk-wlcikcifonhhjetbxffcggckwacddbmedugipfckhpmqsuwh",#hgq
        "sk-nffswispohtowcalrfhmvgihwiljtqwmpgkmzjtrnqcdljcp",#zhangyong
        "sk-cbgvymxlwfhagvohubkbmbivzpmdfkgpgyqzftzdkmjbeygt",#13345560769
        "sk-ezraojkfkxxegsbhuriyrnneeubsokbhhzjvvkfxinhgltqt",#19355265824
        "sk-kzjxgalflvfizcjteviigszbbmqmnsxbcavalctitlmizgqd",#19275604712
        ]
client = OpenAI(
    base_url='https://api.siliconflow.cn/v1',
    api_key=api_key_list[0]
)

# 发送带有流式输出的请求
response = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1",
    messages=[
        {"role": "system","content":"You are a moveable robot in a 2D map, you task is to find the shortest path to the target position."},
        {"role": "user", "content":"""The map is defined by border and lines as fellows,
""
Length,Height=15,8
Width1=2
Width2=0.6
border=[[(Length/2,-Height/2-Width1),(Length/2,-Height/2),(Width2/2,-Height/2),(Width2/2,Height/2),(Length/2,Height/2),(Length/2,Height/2+Width1),(-Length/2,Height/2+Width1),(-Length/2,Height/2),(-Width2/2,Height/2),(-Width2/2,-Height/2),(-Length/2,-Height/2),(-Length/2,-Height/2-Width1),(Length/2,-Height/2-Width1)]]
""
border is a list of lines, each line is a wall that the robot can not pass,
the first and last point in lines should be the same, which means the map is a closed map.

Now your are navigating in this map, your start position is (-6,-Height/2-Width1/2), and the target position is (-6,Height/2+Width1/2), 
you can move to the any directions, but you can not pass the walls defined by the border and lines, you need to find the shortest path to the target position, 
you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible,
the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Give me the shortest path you found, and the path should be output as: ""path=[(x0,y0),(x1,y1),...,(xn,yn)] ""
"""}
    ],
    stream=True  # 启用流式输出
)

# 逐步接收并处理响应
for chunk in response:
    chunk_message = chunk.choices[0].delta.content
    print(chunk_message, end='', flush=True)

NoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNoneNone

In [ ]:
api_key = "sk-rcnkczywequftoobogneniitnbcfjummgnpfhbuxncbifihd"  # 替换为我的硅基流动API密钥


chat1 = DeepSeekChat(api_key)
chat1.add_message("system","Your are Bob, now you are playing a Rock-scissors-paper game with Alice, the rule is Rock beat scissors, scissors beat paper, paper beat Rock. you are only allowed to reply with 'Rock', 'paper' or 'scissors'.When you are prompt with 'let's begin' it mean the game is begin, you should reply 'Rock', 'paper' or 'scissors'. When you are prompt with 'Alice is Rock, paper or scissors',it tell you Alice's choose in last round, and you should reply 'Rock', 'paper' or 'scissors' according to Alice's reply. ")
chat1.add_message("user", "let's begin")
response1 = chat1.get_response()

chat2 = DeepSeekChat(api_key)
chat2.add_message("system","Your are Alice, now you are playing a Rock-scissors-paper game with Bob, the rule is Rock beat scissors, scissors beat paper, paper beat Rock. you are only allowed to reply with 'Rock', 'paper' or 'scissors'.When you are prompt with 'let's begin' it mean the game is begin, you should reply 'Rock', 'paper' or 'scissors'. When you are prompt with 'Alice is Rock, paper or scissors',it tell you Alice's choose in last round, and you should reply 'Rock', 'paper' or 'scissors' according to Alice's reply. ")
chat2.add_message("user", "let's begin")
response2 = chat2.get_response()

SSLError: HTTPSConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /v1 (Caused by SSLError(SSLError(1, '[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:1131)')))

In [ ]:
reply2=f"Bob is {response1}"
reply1=f"Alice is {response2}"
print(reply1,reply2)

In [ ]:
chat1.add_message("user", reply1)
response1 = chat1.get_response()
chat2.add_message("user", reply2)
response2 = chat2.get_response()
reply2=f"Bob is {response1}"
reply1=f"Alice is {response2}"
print(reply1,reply2)

## ollama

In [2]:
from ollama import chat
from ollama import ChatResponse
import re

def filter_think_tags(content):
    pattern = r'<think>.*?</think>'
    filtered_text = re.sub(pattern, '', content, flags=re.DOTALL)
    return filtered_text.strip()
def extract_think_tags(content):
    pattern = re.compile(r"<think>(.*?)</think>", re.DOTALL)
    match = pattern.search(content)
    if match:
        return match.group(1)
    else:
        return "No content found in <think> tags"
      
messages1=[
  {
    'role': 'system',
    'content':"""Your are Bob, now you are playing a Rock-scissors-paper game with Alice, 
    the rule is Rock beat scissors, scissors beat paper, paper beat Rock. 
    When you are prompt with 'let's begin' it mean the game is begin, you should reply 'Rock', 'paper' or 'scissors'. 
    When you are prompt with 'Alice is Rock, paper or scissors',it tell you Alice's choose in last round, 
    and you should reply 'Rock', 'paper' or 'scissors' according to Alice's reply.
    Note: You are only allowed to reply with 'Rock', 'paper' or 'scissors'."""
  }
]
messages1.append({'role': 'user','content': "let's begin",})
response1: ChatResponse = chat(model='qwq:latest', messages=messages1,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
anwser1=filter_think_tags(response1.message.content)
#判断是否符合规则，获取此轮出牌
while anwser1.lower() not in ['rock','paper','scissors']:
    print(f"Bob is {anwser1}")
    response1: ChatResponse = chat(model='qwq:latest', messages=messages1,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
    anwser1=filter_think_tags(response1.message.content)
messages1.append({'role':"assistant", 'content':response1.message.content})

messages2=[
  {
    'role': 'system',
    'content':"""Your are Alice, now you are playing a Rock-scissors-paper game with Bob, 
    the rule is Rock beat scissors, scissors beat paper, paper beat Rock. 
    When you are prompt with 'let's begin' it mean the game is begin, you should reply 'Rock', 'paper' or 'scissors'. 
    When you are prompt with 'Bob is Rock, paper or scissors',it tell you Bob's choose in last round, 
    and you should reply 'Rock', 'paper' or 'scissors' according to Bob's reply.
    Note: You are only allowed to reply with 'Rock', 'paper' or 'scissors'."""
  }
]
messages2.append({'role': 'user','content': "let's begin",})
response2: ChatResponse = chat(model='qwq:latest', messages=messages2,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
anwser2=filter_think_tags(response2.message.content)
#判断是否符合规则，获取此轮出牌
while anwser2.lower() not in ['rock','paper','scissors']:
    print(f"Alice is {anwser2}")
    response2: ChatResponse = chat(model='qwq:latest', messages=messages2,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
    anwser2=filter_think_tags(response2.message.content)
messages2.append({'role':"assistant", 'content':response2.message.content})

reply2=f"Bob is {anwser1}"
reply1=f"Alice is {anwser2}"
print(reply2,reply1)


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [74]:
print(messages2,messages1)

[{'role': 'system', 'content': "Your are Alice, now you are playing a Rock-scissors-paper game with Bob, \n    the rule is Rock beat scissors, scissors beat paper, paper beat Rock. \n    When you are prompt with 'let's begin' it mean the game is begin, you should reply 'Rock', 'paper' or 'scissors'. \n    When you are prompt with 'Bob is Rock, paper or scissors',it tell you Bob's choose in last round, \n    and you should reply 'Rock', 'paper' or 'scissors' according to Bob's reply.\n    Note: You are only allowed to reply with 'Rock', 'paper' or 'scissors'."}, {'role': 'user', 'content': "let's begin"}, {'role': 'assistant', 'content': '<think>\nOkay, so I\'m Alice, and I\'m playing Rock-Paper-Scissors with Bob. The rules are straightforward: Rock beats scissors, scissors beat paper, and paper beats rock. Got it.\n\nThe game starts when the prompt says "let\'s begin." My first move is to choose either Rock, Paper, or Scissors. Since this is the beginning, I don\'t have any information

In [ ]:
messages1.append({'role': 'user','content': reply1,})#提示上一轮对手的出牌
#获取此轮出牌
response1: ChatResponse = chat(model='deepseek-r1:32b', messages=messages1,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
anwser1=filter_think_tags(response1.message.content)
#判断是否符合规则，获取此轮出牌
while anwser1.lower() not in ['rock','paper','scissors']:
    response1: ChatResponse = chat(model='deepseek-r1:32b', messages=messages1,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
    anwser1=filter_think_tags(response1.message.content)
#加入此轮回复消息
messages1.append({'role':"assistant", 'content':response1.message.content})    

messages2.append({'role': 'user','content': reply2,})
response2: ChatResponse = chat(model='deepseek-r1:32b', messages=messages2,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
anwser2=filter_think_tags(response2.message.content)
while anwser2.lower() not in ['rock','paper','scissors']:
    response2: ChatResponse = chat(model='deepseek-r1:32b', messages=messages2,options={"temperature":0.1,'top_p': 0.3,'num_predict': 1024,})
    anwser2=filter_think_tags(response2.message.content)
messages2.append({'role':"assistant", 'content':response2.message.content})
reply2=f"Bob is {anwser1}"
reply1=f"Alice is {anwser2}"
print(reply2,reply1)    
print(messages2[3],messages1[3])

Bob is Scissors Alice is Scissors
{'role': 'user', 'content': 'Bob is Rock'} {'role': 'user', 'content': 'Alice is Rock'}


In [18]:
messages1=[
        {"role": "user", "content":"""
         You are a moveable robot in a 2D map, you task is to find the shortest path to the target position. The map is defined by border and lines as fellows,
""
Length,Height=15,8
Width1=2
Width2=0.6
border=[[(Length/2,-Height/2-Width1),(Length/2,-Height/2),(Width2/2,-Height/2),(Width2/2,Height/2),(Length/2,Height/2),(Length/2,Height/2+Width1),(-Length/2,Height/2+Width1),(-Length/2,Height/2),(-Width2/2,Height/2),(-Width2/2,-Height/2),(-Length/2,-Height/2),(-Length/2,-Height/2-Width1),(Length/2,-Height/2-Width1)]]
""
border is a list of lines, each line is a wall that the robot can not pass.
Now your are navigating in this map, your start position is (-6,-Height/2-Width1/2), and the target position is (-6,Height/2+Width1/2), 
you can move to the any directions, you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible, the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Please reason step by step, and put your final answer within \boxed{}. For example,\boxed{[(x0,y0),...,(xn,yn)]}.
"""}
]
messages2=[
        {"role": "user", "content":"""
         You are a moveable robot in a 2D map, you task is to find the shortest path to the target position. The map is defined by border and lines as fellows,
""
border = [[(-4.5, 4.5), (-4.5, -4.5), (4.5, -4.5), (4.5, 4.5),(-4.5, 4.5)]]
inner_lines = [[(0.85, 1.5), (4.51,1.5)],[(1.5,-0.85), (1.5,-4.51)],[(-0.85, -1.5), (-4.51,-1.5)],[(-1.5,0.85), (-1.5,4.51)]]
""
border and inner_lines are lists of lines, each line is a wall that the robot can not pass.
Now your are navigating in this map, your start position is (-3.5, 3.5), and the target position is (3.5, -3.5), 
you can move to the any directions, you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible, the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Please reason step by step, and put your final answer within \boxed{}. For example,\boxed{[(x0,y0),...,(xn,yn)]}.
"""}
]
messages3=[
        {"role": "user", "content":"""
         You are a moveable robot in a 2D map, you task is to find the shortest path to the target position. The map is defined by border and lines as fellows,
""
border = [[(-8.5, 8.5), (-8.5, -8.5), (8.5, -8.5), (8.5, 8.5), (-8.5, 8.5)]]
inner_lines = [[(-5.968789881006918, 5.221555759840556), (-5.511502133634095, 4.953623089001988), (-4.57121011899308, 6.558444240159443), (-5.028497866365903, 6.826376910998012), (-5.968789881006918, 5.221555759840556)], [(-7.917081412657099, 0.2916636490649136), (-7.6118410705194, -0.293510114471748), (-5.882918587342901, 0.6083363509350885), (-6.188158929480599, 1.1935101144717501), (-7.917081412657099, 0.2916636490649136)], [(-4.181179219068674, -2.460306548509199), (-3.7952381358435416, -2.142423739475726), (-4.818820780931327, -0.8996934514908013), (-5.204761864156459, -1.2175762605242744), (-4.181179219068674, -2.460306548509199)], [(-6.002922450057546, -5.661577398339072), (-5.975596492525679, -6.33101992167096), (-4.197077549942453, -6.258422601660929), (-4.224403507474319, -5.58898007832904), (-6.002922450057546, -5.661577398339072)], [(-1.9823404906998385, -5.248732497085582), (-1.3083833438613055, -5.4720747551230255), (-0.8176595093001617, -3.9912675029144173), (-1.4916166561386943, -3.767925244876974), (-1.9823404906998385, -5.248732497085582)], [(2.0556709388424483, -6.2309391960346785), (2.3844176889219604, -6.144189127953098), (1.8843290611575518, -4.249060803965323), (1.5555823110780402, -4.3358108720469035), (2.0556709388424483, -6.2309391960346785)], [(5.871518838708953, -3.274370047262569), (6.2320768050141435, -3.5600236012993864), (7.368481161291048, -2.125629952737432), (7.007923194985857, -1.8399763987006148), (5.871518838708953, -3.274370047262569)], [(5.5748295724206045, -0.7157636502422351), (5.947158726214549, -0.32464857608235764), (4.665170427579396, 0.8957636502422348), (4.292841273785452, 0.5046485760823574), (5.5748295724206045, -0.7157636502422351)], [(5.515947251547304, 2.01544447568999), (6.084562161693597, 2.262582259040332), (5.084052748452696, 4.564555524310011), (4.5154378383064016, 4.3174177409596695), (5.515947251547304, 2.01544447568999)], [(1.7778602825200456, 5.837376067154208), (2.1716705202382443, 6.303223141491201), (0.9421397174799528, 7.3426239328457905), (0.5483294797617546, 6.876776858508798), (1.7778602825200456, 5.837376067154208)]]
""
border and inner_lines are lists of lines, each line is a wall that the robot can not pass.
Now your are navigating in this map, your start position is (3, -5), and the target position is (-3, -5), 
you can move to the any directions, you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible, the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Please reason step by step, and put your final answer within \boxed{}. For example,\boxed{[(x0,y0),...,(xn,yn)]}.
"""}
]

In [107]:
messages=messages1.copy()
messages.append({'role': 'assistant','content':'hh'})
print(messages1)

[{'role': 'system', 'content': 'You are a moveable robot in a 2D map, you task is to find the shortest path to the target position.'}, {'role': 'user', 'content': 'The map is defined by border and lines as fellows,\n""\nLength,Height=15,8\nWidth1=2\nWidth2=0.6\nborder=[[(Length/2,-Height/2-Width1),(Length/2,-Height/2),(Width2/2,-Height/2),(Width2/2,Height/2),(Length/2,Height/2),(Length/2,Height/2+Width1),(-Length/2,Height/2+Width1),(-Length/2,Height/2),(-Width2/2,Height/2),(-Width2/2,-Height/2),(-Length/2,-Height/2),(-Length/2,-Height/2-Width1),(Length/2,-Height/2-Width1)]]\n""\nborder is a list of lines, each line is a wall that the robot can not pass,\nthe first and last point in lines should be the same, which means the map is a closed map.\n\nNow your are navigating in this map, your start position is (-6,-Height/2-Width1/2), and the target position is (-6,Height/2+Width1/2), \nyou can move to the any directions, but you can not pass the walls defined by the border and lines, you n

In [19]:
from ollama import Client

client = Client(
#     host='http://172.30.54.172:11434',#3090
#     host='http://172.30.165.188:11434',#4090
# host='http://127.0.0.1:11434',
#http://172.30.219.162:11434 #A6000
host='http://172.30.219.162:11434',
    headers={'x-some-header': 'some-value'}
)
# from ollama import chat
chat=client.chat
from ollama import ChatResponse
# deepseek-r1:1.5b,qwq:latest
response: ChatResponse = chat(model='qwq:latest', messages=messages3.copy(),options={"temperature":0.6,'top_p': 0.95,'top_k': 20,'num_predict':-1,'num_ctx': 4096*4},stream=False)
      # 启用流式输出
# 逐步接收并处理响应
# for chunk in response:
#     chunk_message = chunk.message.content
#     print(chunk_message, end='', flush=True)
# print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

<think>

Okay, so I need to find the shortest path from (3, -5) to (-3, -5) in this map with all these walls. Hmm, first, I should probably visualize the environment based on the given borders and inner lines. The border is a square from (-8.5, 8.5) down to (8.5, -8.5), but wait actually looking at the coordinates, maybe it's an axis-aligned square? Let me check:

The border is defined as a polygon with vertices [(-8.5,8.5), (-8.5,-8.5), (8.5,-8.5), (8.5,8.5), back to first]. So yes, that makes a square from -8.5 to 8.5 in both x and y axes. The start is at (3,-5) which is inside the border, target (-3,-5). 

Now the inner lines are lists of points forming walls. Each wall is a polygon? Or each line segment connects consecutive points? Looking at inner_lines[0], for example: it has five points, so maybe it's a closed shape (since last point goes back to first)? The problem states "each line is a wall that the robot cannot pass". So each of these lists represents a closed polygon obstac

In [ ]:
print(response.eval_duration/1e9,response.eval_count,response.eval_count/response.eval_duration*1e9)

358.921 8871 24.715745247561443


In [ ]:
import re

def filter_think_tags(content):
    pattern = r'<think>.*?</think>'
    filtered_text = re.sub(pattern, '', content, flags=re.DOTALL)
    return filtered_text.strip()
def extract_think_tags(content):
    pattern = re.compile(r"<think>(.*?)</think>", re.DOTALL)
    match = pattern.search(content)
    if match:
        return match.group(1)
    else:
        return "No content found in <think> tags"

In [113]:
res_content=filter_think_tags(response.message.content)
# text = '''The shortest path from (-6,-1.6) to (-6,1.6) avoids crossing any borders and only requires three waypoints: start, middle, and end. path=[(-1,-1.6), (-6,0.6), (7.5,0.6), (7.5,1.6)] sdfgrt'''

#直接匹配到path=后面的内容，且是坐标形式
pattern = r'path=\[\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\)(?:\s*,\s*\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\))*\]'
match = re.search(pattern, res_content)
if match:
    print(match.group(0))
else:
    print("No match found")
# extract_think_tags(response.message.content)
import ast
# Input string
path_str = match.group(0)

# Extract the part after "path=" and evaluate it as a Python literal
path_list = ast.literal_eval(path_str.split("=", 1)[1])

print(path_list)
# print(type(path_list))

path=[(-6,-8.5), (-6,-7.5), (-6,-6.5), (-6,-5.5), (-6,-4.5), (-6,-3.5), (-6,-2.5), (-6,0.5), (-6,1.5), (-6,2.5), (-6,3.5), (-6,4.5), (-6,5.5), (-6,6.5), (-6,7.5), (-6,8.5)]
[(-6, -8.5), (-6, -7.5), (-6, -6.5), (-6, -5.5), (-6, -4.5), (-6, -3.5), (-6, -2.5), (-6, 0.5), (-6, 1.5), (-6, 2.5), (-6, 3.5), (-6, 4.5), (-6, 5.5), (-6, 6.5), (-6, 7.5), (-6, 8.5)]


In [ ]:
import re

text = '''The shortest path from (-6,-1.6) to (-6,1.6) avoids crossing any borders and only requires three waypoints: start, middle, and end. path=[(-1,-1.6), (-6,0.6), (7.5,0.6), (7.5,1.6)] sdfgrt'''

#直接匹配到path=后面的内容，且是坐标形式
pattern = r'path=\[\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\)(?:\s*,\s*\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\))*\]'
match = re.search(pattern, text)
if match:
    print(match.group(0))

ddd
path=[(-1,-1.6), (-6,0.6), (7.5,0.6), (7.5,1.6)]


In [1]:
from ollama import Client
import ast
client = Client(
#     host='http://172.30.54.172:11434',#3090
#     host='http://172.30.165.188:11434',#4090
# http://127.0.0.1:11434
#172.30.219.162 A6000
host='http://172.30.219.162:11434',
    headers={'x-some-header': 'some-value'}
)
# from ollama import chat
chat=client.chat
from ollama import ChatResponse
import re
from SimEnv import SimulationEnvironment_Scenario2, checkintersection
env=SimulationEnvironment_Scenario2()
def filter_think_tags(content):
    pattern = r'<think>.*?</think>'
    filtered_text = re.sub(pattern, '', content, flags=re.DOTALL)
    return filtered_text.strip()
def extract_think_tags(content):
    pattern = re.compile(r"<think>(.*?)</think>", re.DOTALL)
    match = pattern.search(content)
    if match:
        return match.group(1)
    else:
        return "No content found in <think> tags"
messages1=[
{"role": "system","content":"You are a moveable robot in a 2D map, you task is to find the shortest path to the target position."},
        {"role": "user", "content":"""The map is defined by border and lines as fellows,
""
Length,Height=15,8
Width1=2
Width2=0.6
border=[[(Length/2,-Height/2-Width1),(Length/2,-Height/2),(Width2/2,-Height/2),(Width2/2,Height/2),(Length/2,Height/2),(Length/2,Height/2+Width1),(-Length/2,Height/2+Width1),(-Length/2,Height/2),(-Width2/2,Height/2),(-Width2/2,-Height/2),(-Length/2,-Height/2),(-Length/2,-Height/2-Width1),(Length/2,-Height/2-Width1)]]
""
border is a list of lines, each line is a wall that the robot can not pass,
the first and last point in lines should be the same, which means the map is a closed map.

Now your are navigating in this map, your start position is (-6,-Height/2-Width1/2), and the target position is (-6,Height/2+Width1/2), 
you can move to the any directions, but you can not pass the walls defined by the border and lines, you need to find the shortest path to the target position, 
you can use any shortest path planning algorithm to solve this problem,the shortest path should be represented by a list of waypoint, each waypoint is a tuple of (x,y), 
the first waypoint should be the start position, and the last waypoint should be the target position, 
You should use waypoint as less as possible,
the line between two adjacent waypoint should not intersect with any wall or broder, otherwise the path is invalid.
Give me the shortest path you found, and the path should be output as: ""path=[(x0,y0),(x1,y1),...,(xn,yn)] ""
"""}
]

In [ ]:

messages=messages1.copy()
for i in range(10):
    print(f"round {i}")
    response: ChatResponse = chat(model='qwq:latest', messages=messages,options={"temperature":0.6,'top_p': 0.95,'top_k': 20,'num_predict':-1,'num_ctx': 4096*4})#deepseek-r1:1.5b,'qwq:latest'
    res_content=filter_think_tags(response.message.content)
    pattern = r'path=\[\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\)(?:\s*,\s*\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\))*\]'
    match = re.search(pattern, res_content)
    if not match:
        print("answer error format")
        messages=messages1.copy()
        continue
    path_str = match.group(0)
    path_list = ast.literal_eval(path_str.split("=", 1)[1])
    if path_list[0]!=(-6,-5) or path_list[-1]!=(-6,5):
        print("the path is not begin with start position or end with goal position")
        messages.append({'role': 'assistant','content':response.message.content})
        messages.append({'role': 'user','content': "the path is not begin with start position or end with goal position",})
        continue
    if checkintersection(path_list,env):
        print("the path is intersect with border or wall")
        messages.append({'role': 'assistant','content':response.message.content})
        messages.append({'role': 'user','content': "the path is intersect with border or wall",})
        continue
    print(f"the path is valid {path_list}")
    



round 0
the path is not begin with start position or end with goal position
round 1
answer error format
round 2
the path is not begin with start position or end with goal position
round 3
the path is not begin with start position or end with goal position
round 4
the path is not begin with start position or end with goal position
round 5
the path is not begin with start position or end with goal position
round 6
the path is not begin with start position or end with goal position
round 7
the path is not begin with start position or end with goal position
round 8
the path is not begin with start position or end with goal position
round 9
the path is not begin with start position or end with goal position


In [ ]:
import re
import ast
from multiprocessing import Pool

# 假设这些函数和变量已经在全局作用域定义
# def chat(...):  # 模型调用函数
# def filter_think_tags(content):  # 内容过滤函数
# def checkintersection(path_list, env):  # 路径检查函数
# messages1 = [...]  # 初始消息列表
# env = ...  # 环境变量
# import os
# os.environ['OLLAMA_NUM_PARALLEL'] = '20'
def process_task(i, messages1):
    messages = messages1.copy()
    print(f"Round {i} processing...")
    
    # 调用模型生成响应
    response = chat(
        model='deepseek-r1:1.5',#deepseek-r1:1.5,qwq:latest
        messages=messages,
        options={"temperature":0.6, 'top_p':0.95, 'top_k':20, 'num_predict':-1, 'num_ctx':4096 * 4}
    )
    res_content = filter_think_tags(response.message.content)
    
    pattern = r'path=\[\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\)(?:\s*,\s*\(\s*-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?\s*\))*\]'
    match = re.search(pattern, res_content)
    if not match:
        print(f"Round {i} error: answer format invalid")
        return {"success": False, "error": "format error", "messages": messages}
    

    path_str = match.group(0)
    path_list = ast.literal_eval(path_str.split("=", 1)[1])

    # 检查起点和终点
    if not (path_list and path_list[0] == (-6, -5) and path_list[-1] == (-6, 5)):
        print(f"Round {i} error: invalid start/end points")
        return {"success": False, "error": "invalid start/end", "messages": messages}
    
    # 检查路径是否与边界相交
    if checkintersection(path_list, env):
        print(f"Round {i} error: path intersects with obstacles")
        return {"success": False, "error": "intersect", "messages": messages}
    
    # 路径有效
    print(f"Round {i} success: {path_list}")
    return {"success": True, "path": path_list}

if __name__ == "__main__":
    with Pool(processes=10) as pool:
        results = pool.starmap(process_task, [(i, messages1) for i in range(10)])
        
        # 统计结果
        success_count = sum(1 for res in results if res["success"])
        print(f"\nTotal rounds: 10")
        print(f"Successful paths: {success_count}")
        for res in results:
            if res["success"]:
                print(f"  Valid path: {res['path']}")

Round 0 processing...Round 6 processing...Round 3 processing...Round 1 processing...Round 4 processing...Round 2 processing...Round 7 processing...Round 5 processing...Round 8 processing...Round 9 processing...









